# Práctica 1: Soluciones de los ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [scikit-learn](https://scikit-learn.org) de Python.

In [87]:
import numpy as np
np.random.seed(982375)

### Ejercicio 1

El fichero `cars.csv` contiene información acerca de la idoneidad de una serie de coches, en función de los siguientes atributos discretos:

* Precio de compra (`buying`): posibles valores `vhigh`, `high`, `med`, `low`.
* Coste de mantenimiento (`maint`): posibles valores `vhigh`, `high`, `med`, `low`.
* Número de puertas (`doors`): posibles valores `2`, `3`, `4`, `5more`.
* Número de asientos (`persons`): posibles valores `2`, `4`, `more`.
* Tamaño del maletero (`lug_boot`): posibles valores `small`, `med`, `big`.
* Nivel de seguridad estimada (`safety`): posibles valores `low`, `med`, `high`.

La idoneidad de cada coche se indica mediante el atributo `acceptability`, que los clasifica como `unacc`, `acc`, `good` o `vgood`.

Se pide realizar lo siguiente:

1. Estimar mediante validación cruzada la tasa de acierto que obtendría un modelo naive Bayes para distintos valores del parámetro de suavizado.
2. Seleccionar el mejor valor de suavizado, entrenar un modelo naive Bayes a partir de todos los ejemplos usados para la validación cruzada y proporcionar su tasa de acierto sobre un conjunto de prueba reservado desde el principio.

In [88]:
import pandas as pd

In [89]:
cars = pd.read_csv('cars.csv')
cars.head()

,buying,maint,doors,persons,lug_boot,safety,acceptability
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [90]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

In [91]:
codificador_atributos = OrdinalEncoder()
atributos = codificador_atributos.fit_transform(cars.loc[:, 'buying':'safety'])
atributos

array([[3., 3., 0., 0., 2., 1.],
       [3., 3., 0., 0., 2., 2.],
       [3., 3., 0., 0., 2., 0.],
       ...,
       [1., 1., 3., 2., 0., 1.],
       [1., 1., 3., 2., 0., 2.],
       [1., 1., 3., 2., 0., 0.]])

In [92]:
codificador_objetivo = LabelEncoder()
objetivo = codificador_objetivo.fit_transform(cars['acceptability'])
objetivo

array([2, 2, 2, ..., 2, 1, 3])

In [2]:
3500 * 0.2, 3500*0.8

(700.0, 2800.0)

In [93]:
from sklearn.model_selection import train_test_split

In [94]:
(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = train_test_split(
    atributos, objetivo,
    test_size=.2,
    stratify=objetivo)

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [95]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import CategoricalNB

In [96]:
# extraer los parametros de CategoricalNB
from inspect import signature
signature(CategoricalNB)

<Signature (*, alpha=1.0, force_alpha=True, fit_prior=True, class_prior=None, min_categories=None)>

In [97]:
# alpha: suavizado de Laplace, para evitar problemas con probabilidades de valor 0
rejilla_de_hiperparámetros = {'alpha': range(1, 6), "force_alpha": [True, False], }

# Por defecto GridSearchCV usa la tasa de acierto como métrica y reentrena
# con todos los ejemplos el mejor modelo encontrado
búsqueda_en_rejilla = GridSearchCV(
    CategoricalNB(),
    rejilla_de_hiperparámetros,
    cv=10,
    #scoring='accuracy',
    #scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'],
)
búsqueda_en_rejilla.fit(atributos_entrenamiento, objetivo_entrenamiento)
mejor_modelo = búsqueda_en_rejilla.best_estimator_
mejor_modelo.score(atributos_prueba, objetivo_prueba)

0.815028901734104

In [98]:
búsqueda_en_rejilla.cv_results_

{'mean_fit_time': array([0.00456276, 0.00376019, 0.00394208, 0.00258532, 0.00204849]),
 'std_fit_time': array([1.17419650e-03, 4.40674900e-04, 5.31313105e-04, 7.37403167e-04,
        7.87275333e-05]),
 'mean_score_time': array([0.00138214, 0.00118375, 0.0012809 , 0.0007561 , 0.00060093]),
 'std_score_time': array([1.94890482e-04, 1.20575326e-04, 3.12354131e-04, 2.26918108e-04,
        3.75368455e-05]),
 'param_alpha': masked_array(data=[1, 2, 3, 4, 5],
              mask=[False, False, False, False, False],
        fill_value=999999),
 'params': [{'alpha': 1},
  {'alpha': 2},
  {'alpha': 3},
  {'alpha': 4},
  {'alpha': 5}],
 'split0_test_score': array([0.87769784, 0.8705036 , 0.8705036 , 0.8705036 , 0.8705036 ]),
 'split1_test_score': array([0.83453237, 0.83453237, 0.83453237, 0.82733813, 0.82733813]),
 'split2_test_score': array([0.85507246, 0.83333333, 0.83333333, 0.83333333, 0.83333333]),
 'split3_test_score': array([0.85507246, 0.85507246, 0.85507246, 0.84782609, 0.84782609]),
 'sp

In [99]:
búsqueda_en_rejilla.best_params_

{'alpha': 1}

### Ejercicio 2

Los púlsares son un tipo raro de estrella de neutrones que produce emisiones de radio detectables aquí en la Tierra. Son de considerable interés científico como sondas del espacio-tiempo, el medio interestelar y los estados de la materia.

A medida que los púlsares giran, su haz de emisión recorre el cielo y, cuando cruza nuestra línea de visión, produce un patrón detectable de emisión de radio de banda ancha. Como los púlsares giran rápidamente, este patrón se repite periódicamente. Por tanto, la búsqueda de púlsares implica buscar señales de radio periódicas con grandes radiotelescopios.

Cada púlsar produce un patrón de emisión algo diferente, que varía levemente con cada rotación. Por lo tanto, una detección de señal potencial conocida como «candidata» se promedia a lo largo de muchas rotaciones del púlsar, según lo determinado por la duración de una observación. A falta de información adicional, cada candidato podría describir un púlsar real. Sin embargo, en la práctica, casi todas las detecciones son causadas por interferencias de radiofrecuencia (RFI) y ruido, lo que dificulta encontrar señales legítimas.

El fichero `pulsar_stars.csv` contiene datos acerca de una serie de púlsares reales y de ejemplos espurios producidos por RFI y ruido. Cada candidato se describe mediante ocho atributos continuos extraídos de las señales recibidas.

Se pide realizar lo siguiente:

1. Dividir el conjunto de ejemplos en un subconjunto de entrenamiento (80&nbsp;% de los ejemplos) y un subconjunto de prueba (20&nbsp;% de los ejemplos). La división debe realizarse mediante muestreo estratificado, ya que la cantidad de ejemplos que se corresponden con púlsares reales es mucho menor que la de los que son interferencias y ruido.
2. Construir un árbol de decisión a partir del subconjunto de entrenamiento y calcular la matriz de confusión sobre el conjunto de prueba para cada combinación de los siguientes valores:
   - Máxima profundidad del árbol (argumento `max_depth`): de 1 a 5.
   - Cantidad mínima de ejemplos en las hojas (argumento `min_samples_leaf`): 1, 3 y 5.
   - Cantidad mínima de ejemplos para poder particionar (argumento `min_samples_split`): 10, 15 y 20.
3. De entre los árboles construidos en el apartado anterior seleccionar uno con máxima tasa de acierto sobre el conjunto de prueba, uno con máxima sensibilidad y uno con máxima precisión.

**Ayuda**: para los apartados 2 y 3 considerar el uso de [`ParameterGrid`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ParameterGrid.html) del módulo `model_selection`.

### Ejercicio 3

El hormigón es el material más importante en la ingeniería civil. La resistencia a la compresión del hormigón es una función altamente no lineal de su edad y sus ingredientes.

El fichero `concrete_data.csv` contiene la siguiente información acerca de diferentes muestras de hormigón:

* Contenido de cemento (`Cement`).
* Contenido de escoria de alto horno (`Blast Furnace Slag`).
* Contenido de cenizas volantes (`Fly Ash`).
* Contenido de agua (`Water`).
* Contenido de superplastificantes (`Superplasticizer`).
* Contenido de agregados gruesos (`Coarse Aggregate`).
* Contenido de agregados finos (`Fine Aggregate`).
* Edad del hormigón (`Age`).

El objetivo es predecir la resistencia a la compresión (`Strength`) a partir de esos atributos continuos.

Se pide realizar lo siguiente:

1. Definir una tubería que concatene un transformador de columnas que normalice los atributos al intervalo $[0, 1]$ y un modelo $k$NN para regresión.
2. Realizar una búsqueda en rejilla para estimar mediante validación cruzada el coeficiente de determinación obtenido al aplicar la tubería a cada combinación de los valores 1 a 5 para el número de vecinos y las distancias manhattan y euclídea para la métrica.
3. Repetir los pasos 1 y 2 usando ahora la tipificación (es decir, restar la media y dividir por la desviación típica) como procedimiento de normalización de los atributos.
4. Seleccionar el mejor procedimiento de normalización, el mejor valor para el número de vecinos y la mejor métrica y entrenar un modelo $k$NN a partir de todos los ejemplos usados para la validación cruzada, proporcionando finalmente su coeficiente de determinación sobre un conjunto de prueba reservado desde el principio.

**Ayuda**: las clases [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) y [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) del módulo `preprocessing` implementan los procedimientos de normalización, mientras que la clase [`KNeighborsRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) del módulo `neighbors` implementa el modelo $k$NN para una tarea de regresión.

### Ejercicio 4

Los [abulones](https://es.wikipedia.org/wiki/Haliotis) son una familia de moluscos gasterópodos. La edad de cada individuo está correlacionada con el número de anillos de su concha y, por tanto, puede determinarse cortando la concha a través del cono, tiñéndola y contando el número de anillos a través de un microscopio. Este procedimiento requiere mucho tiempo y es propenso a errores, por lo que sería preferible poder determinar la edad directamente a partir de medidas físicas más fáciles de obtener.

El fichero `abalone.csv` contiene la siguiente información de distintos individuos de abulones:

* Sexo (`Sex`): atributo discreto con posibles valores `M` (macho), `F` (hembra) e `I` (infante).
* Longitud (`Length`) en milímetros.
* Diámetro (`Diameter`) en milímetros.
* Altura (`Height`) en milímetros.
* Peso total (`Whole_weight`) en gramos.
* Peso sin la concha (`Shucked_weight`) en gramos.
* Peso intestinal (`Viscera_weight`) en gramos.
* Peso de la concha (`Shell_weight`) en gramos.

Se pide construir el mejor modelo posible para resolver la tarea de predecir el número de anillos (`Rings`) a partir de los atributos anteriores (entonces bastaría sumar 1.5 a ese número de anillos para obtener la edad, en años, del individuo).